<p align="center">
<a href="https://duckietown.com"><img src="../assets/images/dtlogo.png" alt="Duckietown Logo" width="50%"></a>
</p>


<style>
.lx-table {
  margin: 1.5em auto;
  text-align: left;
}

.lx-table caption {
  caption-side: top;
  font-size: 0.9em;
  margin-bottom: 0.6em;
  text-align: center;
}

.lx-figure {
  margin: 1.5em auto;
  text-align: center;
}

.lx-figure figcaption {
  font-size: 0.9em;
  margin-top: 0.6em;
  text-align: center;
}

table {
  margin: 0 auto 1.5em;
}

p:has(> a[id^='table-']) {
  margin: 0;
}

p:has(> a[id^='table-']) + p,
a[id^='table-'] + p {
  font-size: 0.9em;
  margin: 1.5em 0 0.6em;
  text-align: center;
}

p.lx-figure {
  margin: 1.5em auto 0;
}

p.lx-figure + p {
  font-size: 0.9em;
  margin: 0.6em 0 1.5em;
  text-align: center;
}
</style>


# ROS 2 Discovery, Domains, and Names

Two correctly written nodes can still fail to exchange messages if their discovery settings or resolved names differ. Understanding domains, middleware, and names helps you distinguish a configuration mismatch from a problem in the node code. We will check the local shell's settings to see how it joins a graph, then resolve names so you can identify the node or topic you intend to inspect.

## Discovery through middleware

Middleware is the software layer that lets separate ROS 2 nodes find one another and exchange messages. Compatible participants must be able to reach one another over the network and use the same domain identifier (ID). The `ROS_DOMAIN_ID` environment variable contains that identifier, a number that separates independent ROS 2 graphs sharing a network. Participants join the graph associated with their domain ID.

The Duckiedrone uses [Zenoh](https://zenoh.io/docs/overview/what-is-zenoh/) as its ROS 2 middleware. Zenoh is the messaging layer that lets compatible ROS 2 nodes discover one another and exchange data between ROS 2 containers on the Duckiedrone. [`rmw_zenoh_cpp`](https://github.com/ros2/rmw_zenoh) is the ROS 2 adapter that lets ROS 2 use Zenoh. The `RMW_IMPLEMENTATION` environment variable selects that adapter. A `zenoh-router` container connects the ROS 2 containers, and standard ROS 2 services on a Duckiedrone use `ROS_DOMAIN_ID=42`.

### Try it

In the supplied ROS 2 exercise terminal, source ROS 2 and inspect the shell's domain and middleware settings and the nodes it can see. Record the values so you can compare them when opening another terminal.

```bash
source /opt/ros/${ROS2_DISTRO}/setup.bash
printf 'ROS_DOMAIN_ID=%s\n' "${ROS_DOMAIN_ID:-unset}"
printf 'RMW_IMPLEMENTATION=%s\n' "${RMW_IMPLEMENTATION:-unset}"
ros2 node list
```

ROS 2 can use its defaults for settings shown as `unset`. The node list shows the graph visible from this terminal and may be empty before starting a local node. Which settings would you compare to check that two exercise terminals can join the same graph?

<details>
<summary>Interpret the result</summary>
<p>Compare the domain and middleware configuration in both terminals, along with their network connectivity. Matching settings let compatible local nodes discover one another. The printed values provide a reference for that comparison.</p>
</details>

A shell that inspects the Duckiedrone's graph needs its network connection, middleware implementation, and domain ID. Changing the Duckiedrone's domain or middleware can disconnect its running nodes from one another.

## Absolute, relative, and private names

The form of a name affects where it appears in the graph, as [Table 1](#table-1) shows. These examples illustrate name resolution for a hypothetical node named `/DUCKIEDRONE_NAME/camera_node`, where `DUCKIEDRONE_NAME` is the name of your Duckiedrone.

<table id="table-1" class="lx-table" style="margin:1.5em auto; text-align:left;">
  <caption style="caption-side:top; font-size:0.9em; margin-bottom:0.6em; text-align:center;">Table 1: ROS 2 name forms and their resolution.</caption>
  <thead>
    <tr>
      <th>Form</th>
      <th>Example</th>
      <th>Meaning</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <td>Absolute</td>
      <td><code>/DUCKIEDRONE_NAME/camera_node/camera_info</code></td>
      <td>The complete graph name, starting at the root <code>/</code></td>
    </tr>
    <tr>
      <td>Relative</td>
      <td><code>camera_info</code></td>
      <td>ROS 2 adds the node's namespace. For the example node, this becomes <code>/DUCKIEDRONE_NAME/camera_info</code></td>
    </tr>
    <tr>
      <td>Private</td>
      <td><code>~/state</code></td>
      <td>ROS 2 adds the node's namespace and name. For that same node, this becomes <code>/DUCKIEDRONE_NAME/camera_node/state</code></td>
    </tr>
  </tbody>
</table>

Private names are useful for a node's implementation details because they avoid collisions with another instance of the same node. `ros2 topic list -t` shows the actual camera and ToF topic names and types. MAVROS uses topics such as `/mavros/state`; `/mavros/imu/data` is available when the relevant plugins are enabled. The running graph determines which names and types an application can use.

## If you have seen ROS 1

[Table 2](#table-2) translates familiar ROS 1 tools into the ROS 2 tools used here, showing how to accomplish the same jobs with the current middleware and build system.

<table id="table-2" class="lx-table" style="margin:1.5em auto; text-align:left;">
  <caption style="caption-side:top; font-size:0.9em; margin-bottom:0.6em; text-align:center;">Table 2: ROS 1 concepts and their ROS 2 counterparts in this LX.</caption>
  <thead>
    <tr>
      <th>ROS 1</th>
      <th>ROS 2 in this LX</th>
      <th>Practical change</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <td><code>roscore</code> and a ROS master</td>
      <td>Middleware discovery, using Zenoh on Duckiedrones, and <code>ROS_DOMAIN_ID</code></td>
      <td>Middleware handles discovery between compatible participants</td>
    </tr>
    <tr>
      <td><code>catkin_make</code> and <code>devel/setup.bash</code></td>
      <td><code>colcon build</code> and <code>install/setup.bash</code></td>
      <td>Build the workspace, then source its installed overlay</td>
    </tr>
    <tr>
      <td><code>rospy</code> and <code>rospy.Rate</code></td>
      <td><code>rclpy</code>, nodes, timers, and executors</td>
      <td>Timers and subscriptions invoke callbacks while <code>rclpy.spin()</code> runs</td>
    </tr>
    <tr>
      <td><code>rosrun</code> and <code>rostopic</code></td>
      <td><code>ros2 run</code> and <code>ros2 topic</code></td>
      <td>The graph-inspection goal is the same, but the commands differ</td>
    </tr>
    <tr>
      <td><code>anonymous=True</code></td>
      <td>Explicit names and command-line remapping</td>
      <td>Choose a unique name or remap one when starting another node</td>
    </tr>
  </tbody>
</table>

## Further reading

The ROS 2 design article on [topic and service names](https://design.ros2.org/articles/topic_and_service_names.html) explains name resolution. The ROS 2 Jazzy documentation on [`ROS_DOMAIN_ID`](https://docs.ros.org/en/jazzy/Concepts/Intermediate/About-Domain-ID.html) and [QoS settings](https://docs.ros.org/en/jazzy/Concepts/Intermediate/About-Quality-of-Service-Settings.html) expands on discovery and delivery settings. The Duckiedrone manual's [ROS 2 reference](https://docs.duckietown.com/ente/opmanual-dd24/60-software-architecture/ros.html) describes the current containers and interfaces.

## Checkpoint

Run the self-check in the next cell. Write or select a response before revealing the answer.


In [ ]:
import sys
from pathlib import Path

working_directory = Path.cwd()
parent_directory = working_directory.parent
if (parent_directory / "packages").is_dir():
    parent_directory_path = str(parent_directory)
    sys.path.insert(0, parent_directory_path)

from packages.checkpoint_self_check import display_checkpoint_self_checks

display_checkpoint_self_checks()
